In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
secondary_mushroom = fetch_ucirepo(id=848)

# data (as pandas dataframes)
X = secondary_mushroom.data.features
y = secondary_mushroom.data.targets

# metadata
print(secondary_mushroom.metadata)

# variable information
print(secondary_mushroom.variables)

mushroom_data = pd.concat([X, y], axis=1)

target_col = "class"

CONTINUOUS_COLS = ["cap-diameter", "stem-height", "stem-width"]
CATEGORICAL_COLS = [
    col for col in mushroom_data.columns if col not in CONTINUOUS_COLS
]

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

# Fast dev mode: fewer epochs for generators only (set False for full paper run)
FAST_MODE = True
RUN_QUALITY_EVAL = True
TabDDPM_EPOCHS = 10 if FAST_MODE else 150
WGAN_EPOCHS = 10 if FAST_MODE else 100
SDV_EPOCHS = 10 if FAST_MODE else 300
EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
# All 6 synthetic generators for TSTR evaluation
GENERATORS_TO_EVAL = [
    "CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "ForestDiffusion", "TabDDPM"
]

# Stratified subsample (full dataset has 61069 rows)
_, mushroom_data = train_test_split(
    mushroom_data,
    train_size=N_SAMPLES,
    stratify=mushroom_data[target_col],
    random_state=SEED,
)
mushroom_data = mushroom_data.reset_index(drop=True)

# Handle missing values
numeric_cols = mushroom_data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = [col for col in CATEGORICAL_COLS if col in mushroom_data.columns]

for col in numeric_cols:
    mushroom_data[col] = pd.to_numeric(mushroom_data[col], errors="coerce")
    mushroom_data[col] = mushroom_data[col].fillna(mushroom_data[col].median())

for col in categorical_cols:
    fill = mushroom_data[col].mode().iloc[0] if not mushroom_data[col].mode().empty else "missing"
    mushroom_data[col] = mushroom_data[col].fillna(fill)

# Label-encode categorical columns (ordinal encoding)
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    mushroom_data[col] = le.fit_transform(mushroom_data[col].astype(str))
    label_encoders[col] = le

# Encoded positive class for binary metrics (poisonous = "p")
pos_label = int(label_encoders[target_col].transform(["p"])[0])

print(f"Encoded {len(categorical_cols)} categorical columns.")
print(f"{target_col} mapping: {dict(zip(label_encoders[target_col].classes_, range(len(label_encoders[target_col].classes_))))}")
print(f"pos_label (encoded poisonous class): {pos_label}")

X = mushroom_data.drop(columns=[target_col])
y = mushroom_data[target_col]

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(mushroom_data)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


{'uci_id': 848, 'name': 'Secondary Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/848/secondary+mushroom+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/848/data.csv', 'abstract': 'Dataset of simulated mushrooms for binary classification into edible and poisonous.', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 61068, 'num_features': 20, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2021, 'last_updated': 'Wed Apr 10 2024', 'dataset_doi': '10.24432/C5FP5Q', 'creators': ['Dennis Wagner', 'D. Heider', 'Georges Hattab'], 'intro_paper': {'ID': 259, 'type': 'NATIVE', 'title': 'Mushroom data creation, curation, and simulation to support classification tasks', 'authors': 'Dennis Wagner, D. Heider, Georges Hattab', 'venue': 'Scientific Reports', 'year': 2021, 'journal': None, '

In [3]:
# ---------------------------------------------------
# SINGLE RUN
# ---------------------------------------------------

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

CONTINUOUS_COLS = ["cap-diameter", "stem-height", "stem-width"]
CATEGORICAL_COLS = [
    col for col in mushroom_data.columns if col not in CONTINUOUS_COLS
]

# Re-apply encoding if the load cell was not re-run after edits
if mushroom_data[target_col].dtype == "object":
    numeric_cols = mushroom_data.select_dtypes(include=["int64", "float64"]).columns
    for col in numeric_cols:
        mushroom_data[col] = pd.to_numeric(mushroom_data[col], errors="coerce")
        mushroom_data[col] = mushroom_data[col].fillna(mushroom_data[col].median())
    for col in CATEGORICAL_COLS:
        fill = mushroom_data[col].mode().iloc[0] if not mushroom_data[col].mode().empty else "missing"
        mushroom_data[col] = mushroom_data[col].fillna(fill)
    label_encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        mushroom_data[col] = le.fit_transform(mushroom_data[col].astype(str))
        label_encoders[col] = le
    pos_label = int(label_encoders[target_col].transform(["p"])[0])
elif "label_encoders" not in globals():
    label_encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        le.fit(mushroom_data[col].astype(str))
        label_encoders[col] = le
    try:
        pos_label = int(label_encoders[target_col].transform(["p"])[0])
    except ValueError:
        pos_label = int(mushroom_data[target_col].max())

# ---------------------------------------------------
# GENERATOR TRAINING DATA (no stratified split)
# ---------------------------------------------------

train_real = mushroom_data.copy()
test_real = mushroom_data.copy()

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')



================ SINGLE RUN ================
Training TabDDPM...
[0]
22
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(22)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.3491 Sum: 0.3491
Step 1000/1000 MLoss: 0.0 GLoss: 0.3029 Sum: 0.3029
mlp
Sample timestep    0
Discrete cols: [1, 2, 3, 4, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Num shape:  (1000, 20)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 163.55it/s]|
Column Shapes Score: 71.18%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:01<00:00, 142.68it/s]|
Column Pair Trends Score: 57.92%

Overall Score (Average): 64.55%

TabDDPM: 0.6455


In [4]:
# ForestDiffusion
if 'ForestDiffusion' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training ForestDiffusion...')
        synthetic_forestdiffusion = train_forestdiffusion(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SAMPLES,
            seed=seed,
            fast_mode=FAST_MODE,
        )
        synthetic_datasets['ForestDiffusion'] = synthetic_forestdiffusion.copy()
        print('ForestDiffusion: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_metadata,
            )
            scores['ForestDiffusion'] = quality.get_score()
            print('ForestDiffusion:', round(scores['ForestDiffusion'], 4))
        else:
            print('ForestDiffusion: trained (quality eval skipped)')
    except Exception as e:
        print('ForestDiffusion Failed (training/sampling):')
        traceback.print_exc()
    if 'ForestDiffusion' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)')


Training ForestDiffusion...
ForestDiffusion: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 21/21 [00:00<00:00, 128.86it/s]|
Column Shapes Score: 96.34%

(2/2) Evaluating Column Pair Trends: |██████████| 210/210 [00:01<00:00, 136.60it/s]|
Column Pair Trends Score: 91.06%

Overall Score (Average): 93.7%

ForestDiffusion: 0.937


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

if FAST_MODE:
    models = {
        "LogReg": LogisticRegression(max_iter=500, solver="liblinear", random_state=42),
        "SVM-RBF": LinearSVC(max_iter=500, dual="auto", random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42, max_depth=12),
        "RandomForest": RandomForestClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "GradientBoost": GradientBoostingClassifier(
            n_estimators=30, random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(n_estimators=30, random_state=42),
        "MLP": MLPClassifier(max_iter=200, random_state=42),
    }
else:
    models = {
        "LogReg": LogisticRegression(max_iter=5000, solver="liblinear", random_state=42),
        "SVM-RBF": SVC(kernel="rbf", cache_size=1000, tol=1e-3, random_state=42),
        "KNN": KNeighborsClassifier(n_jobs=-1),
        "NaiveBayes": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "MLP": MLPClassifier(max_iter=500, random_state=42),
    }

print(f"Classifier evaluation: {len(models)} models, {len(EVAL_SEEDS)} seeds")
if FAST_MODE:
    print("FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.")

Classifier evaluation: 10 models, 10 seeds
FAST_MODE: SVM-RBF uses LinearSVC (linear kernel) for speed.


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd


In [8]:
# TRTR is evaluated in the comparison cell below via evaluate_models().
print(
    "Skipping duplicate TRTR cell. "
    f"Run the comparison cell for TRTR/TSTR ({len(models)} models, {len(EVAL_SEEDS)} seeds)."
)


Skipping duplicate TRTR cell. Run the comparison cell for TRTR/TSTR (10 models, 10 seeds).


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np


def _safe_stratify(y):
    y = pd.Series(y).reset_index(drop=True)
    if y.nunique() < 2 or y.value_counts().min() < 2:
        return None
    return y


def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None,
):
    if seeds is None:
        seeds = EVAL_SEEDS

    results = []

    for name, model in models.items():
        print(f"  {name}...", flush=True)

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_train),
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=_safe_stratify(y_test),
            )

            scaler = StandardScaler().fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)
            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)
            if hasattr(clf, "n_jobs"):
                clf.set_params(n_jobs=-1)

            clf.fit(X_train_s, y_train)
            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(accuracy_score(y_test, y_pred))
            f1_scores.append(
                f1_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            precision_scores.append(
                precision_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )
            recall_scores.append(
                recall_score(y_test, y_pred, pos_label=pos_label, average="binary", zero_division=0)
            )

        results.append({
            "Model": name,
            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),
            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),
            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),
            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),
            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} ± {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by="Accuracy Mean", ascending=False)


In [10]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = "class"

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "ForestDiffusion",
    "TabDDPM"
]

seeds = EVAL_SEEDS

print("TRTR (Train Real, Test Real)")
print(
    f"Classifiers: {len(models)} | Seeds: {len(seeds)} | "
    f"Generators: {len(model_order)}"
)
print(f"Classifier models: {list(models.keys())}")
print(f"Synthetic generators: {model_order}")

trtr_results = evaluate_models(
    train_df=mushroom_data,
    test_df=mushroom_data,
    label_col="class",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"Skipping {synth_name} — not in synthetic_datasets")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=mushroom_data,
        label_col="class",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)
Classifiers: 10 | Seeds: 10 | Generators: 6
Classifier models: ['LogReg', 'SVM-RBF', 'KNN', 'NaiveBayes', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoost', 'AdaBoost', 'MLP']
Synthetic generators: ['CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'ForestDiffusion', 'TabDDPM']
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0000,1.0000 ± 0.0001
5,RandomForest,0.9999 ± 0.0000,0.9999 ± 0.0000,0.9999 ± 0.0001,0.9999 ± 0.0001
9,MLP,0.9998 ± 0.0001,0.9998 ± 0.0001,0.9998 ± 0.0001,0.9998 ± 0.0002
2,KNN,0.9994 ± 0.0002,0.9994 ± 0.0002,0.9994 ± 0.0002,0.9994 ± 0.0002
4,DecisionTree,0.9431 ± 0.0031,0.9486 ± 0.0027,0.9504 ± 0.0048,0.9469 ± 0.0043
7,GradientBoost,0.8177 ± 0.0095,0.8351 ± 0.0099,0.8381 ± 0.0070,0.8325 ± 0.0191
0,LogReg,0.6533 ± 0.0034,0.7102 ± 0.0025,0.6623 ± 0.0032,0.7656 ± 0.0028
8,AdaBoost,0.6530 ± 0.0091,0.6587 ± 0.0100,0.7262 ± 0.0216,0.6037 ± 0.0233
1,SVM-RBF,0.6527 ± 0.0035,0.7112 ± 0.0025,0.6603 ± 0.0033,0.7705 ± 0.0027
3,NaiveBayes,0.6094 ± 0.0022,0.5472 ± 0.0039,0.7669 ± 0.0027,0.4253 ± 0.0046


Skipping CTGAN — not in synthetic_datasets
Skipping CopulaGAN — not in synthetic_datasets
Skipping TVAE — not in synthetic_datasets
Skipping GaussianCopula — not in synthetic_datasets
ForestDiffusion - TSTR
  LogReg...
  SVM-RBF...
  KNN...
  NaiveBayes...
  DecisionTree...
  RandomForest...
  ExtraTrees...
  GradientBoost...
  AdaBoost...
  MLP...


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,0.9140 ± 0.0051,0.9234 ± 0.0048,0.9128 ± 0.0043,0.9343 ± 0.0089
5,RandomForest,0.8866 ± 0.0053,0.9020 ± 0.0047,0.8663 ± 0.0061,0.9408 ± 0.0085
2,KNN,0.8545 ± 0.0060,0.8658 ± 0.0058,0.8870 ± 0.0060,0.8456 ± 0.0086
9,MLP,0.8373 ± 0.0079,0.8574 ± 0.0068,0.8348 ± 0.0102,0.8815 ± 0.0111
7,GradientBoost,0.7879 ± 0.0119,0.8152 ± 0.0126,0.7888 ± 0.0069,0.8439 ± 0.0245
4,DecisionTree,0.7803 ± 0.0237,0.8035 ± 0.0212,0.7978 ± 0.0229,0.8096 ± 0.0253
0,LogReg,0.6629 ± 0.0057,0.6988 ± 0.0056,0.6931 ± 0.0053,0.7046 ± 0.0084
1,SVM-RBF,0.6627 ± 0.0056,0.6988 ± 0.0054,0.6925 ± 0.0055,0.7053 ± 0.0078
8,AdaBoost,0.6570 ± 0.0109,0.7055 ± 0.0059,0.6750 ± 0.0201,0.7405 ± 0.0278
3,NaiveBayes,0.5940 ± 0.0043,0.5259 ± 0.0079,0.7471 ± 0.0075,0.4058 ± 0.0094


ForestDiffusion - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
3,NaiveBayes,0.5129 ± 0.0130,0.6409 ± 0.0214,0.5421 ± 0.0074,0.7863 ± 0.0610
0,LogReg,0.5086 ± 0.0137,0.5999 ± 0.0221,0.5469 ± 0.0095,0.6659 ± 0.0480
1,SVM-RBF,0.5085 ± 0.0141,0.5992 ± 0.0224,0.5470 ± 0.0099,0.6640 ± 0.0482
4,DecisionTree,0.4906 ± 0.0306,0.5376 ± 0.0406,0.5410 ± 0.0273,0.5367 ± 0.0595
2,KNN,0.4855 ± 0.0111,0.5407 ± 0.0184,0.5356 ± 0.0092,0.5467 ± 0.0327
7,GradientBoost,0.4827 ± 0.0188,0.5697 ± 0.0323,0.5283 ± 0.0128,0.6208 ± 0.0657
8,AdaBoost,0.4826 ± 0.0133,0.5632 ± 0.0350,0.5292 ± 0.0093,0.6064 ± 0.0751
9,MLP,0.4774 ± 0.0132,0.5330 ± 0.0231,0.5283 ± 0.0102,0.5388 ± 0.0405
6,ExtraTrees,0.4557 ± 0.0151,0.5251 ± 0.0297,0.5083 ± 0.0122,0.5449 ± 0.0528
5,RandomForest,0.4495 ± 0.0214,0.5121 ± 0.0360,0.5028 ± 0.0171,0.5236 ± 0.0610


TabDDPM - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,TabDDPM,ExtraTrees,0.544332,0.474853,0.491706,0.455077,1.0000 ± 0.0000,0.4557 ± 0.0151
1,TabDDPM,RandomForest,0.550375,0.487836,0.497131,0.476256,0.9999 ± 0.0000,0.4495 ± 0.0214
2,TabDDPM,MLP,0.522366,0.466854,0.471505,0.461032,0.9998 ± 0.0001,0.4774 ± 0.0132
3,TabDDPM,KNN,0.513900,0.458748,0.463858,0.452767,0.9994 ± 0.0002,0.4855 ± 0.0111
4,TabDDPM,DecisionTree,0.452497,0.411001,0.409359,0.410214,0.9431 ± 0.0031,0.4906 ± 0.0306
5,TabDDPM,GradientBoost,0.335017,0.265427,0.309747,0.211654,0.8177 ± 0.0095,0.4827 ± 0.0188
6,TabDDPM,LogReg,0.144673,0.110285,0.115387,0.099685,0.6533 ± 0.0034,0.5086 ± 0.0137
7,TabDDPM,AdaBoost,0.170451,0.095423,0.197002,-0.002730,0.6530 ± 0.0091,0.4826 ± 0.0133
8,TabDDPM,SVM-RBF,0.144190,0.111987,0.113372,0.106465,0.6527 ± 0.0035,0.5085 ± 0.0141
9,TabDDPM,NaiveBayes,0.096471,-0.093731,0.224801,-0.360987,0.6094 ± 0.0022,0.5129 ± 0.0130


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
0,ForestDiffusion,0.069099,0.061384,0.070814,0.053172
1,TabDDPM,0.347427,0.278868,0.329387,0.230943


In [11]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
